## 0. 今日の量子コンピュータの問題

- Noisy Intermediate-Scale Quantum (NISQ) デバイス
    - 量子回路が深くなる（ゲート数が多くなる）ほど、誤差が大きくなる
    - 十分な量子ビット数ではない
- 量子デバイスは特別なゲート演算のみが用意されている
- 特定のqubits間の量子ビット演算(multi qubit operation)しか用意されていない
- それぞれの量子デバイスに対して、量子ソフトウェアツールキットが用意されてる


### 0-1. TKETとは
- Quantum Software Development Kit
- C++で実装
- pythonモジュール　`pytket`で利用可能
- 最適化コンパイラ：　ユーザーフレンドリーな回路→量子デバイスで実行可能な回路に変換可能
    - Language-agnostic (多くの量子プログラミングフレームワーク(qiskit, Cirq, etc)をサポート)
    - Retagetable (多くの量子デバイス(Quantinuum, IBM, etc)をサポート)
    - Circuit Optimisation (量子計算時に生じるデバイスエラーの影響を最小化。デバイス依存＆デバイス非依存のものが実装)  

#### H2にはpytketで記述した量子回路をNEXUS経由（qnexusを利用）して実行。  
<img src="./fig/tket.png" width="350">

#### HeliosにはGuppy（New Software for quantum computing）で記述した量子回路をNEXUS経由（qnexusを利用）して実行。  
<img src="./fig/guppy.png" width="350">



#### 参照
- [pytket API ドキュメント](https://docs.quantinuum.com/tket/api-docs/)
- [pytket ユーザーガイド](https://docs.quantinuum.com/tket/user-guide/index.html)
- [qnexus API ドキュメント](https://docs.quantinuum.com/nexus/nexus_api/qnexus_api.html)
- [t|ket⟩ : A Retargetable Compiler for NISQ Devices](https://arxiv.org/abs/2003.10611)

### 0-2. このノートブックで必要となる python パッケージ
Python Python 3.12.11で動作確認をしています。

|  パッケージ （version） |  概要  |
| :---- | :---- |
|  pytket (2.18.1) |  TKETを利用するためのpython モジュール  ( available for python 3.10 or higher )|
|  qnexus (0.48.2) |  Nexusにアクセスし, 量子回路のコンパイルやQuantinuum Hardware/Emulatorへの実行を可能にするpackage  |


In [1]:
!pip freeze |grep pytket

pytket==2.18.1
pytket-qir==2.0.0
pytket-quantinuum==0.59.1


In [2]:
!pip freeze |grep qnexus

qnexus==0.48.2


環境にインストールされていない場合は、以下のセルの＃を取り除き、インストールしてください。

In [3]:
#!pip install -U pytket #TKET量子回路の作成、量子回路の最適化をじっこうするためのパッケージ
#!pip install -U qnexu #Nexusにアクセスし, 量子回路のコンパイルやQuantinuum Hardware/Emulatorへの実行を可能にするパッケージ
#!pip install -U pylatexenc #可視化のためのパッケージ

## 1. TKET量子回路を作成し、可視化する

In [4]:
from pytket import Circuit
from pytket.circuit.display import render_circuit_jupyter

bell = Circuit(2)
bell.H(0).CX(0,1)
bell.measure_all()
render_circuit_jupyter(bell)

## 2. `qnexus`を利用し`TKET`の量子回路を量子デバイス/シミュレータで実行

### 2-1. `qnexus`にログインし、projects/teamsを確認。projectsを指定/作成し、アクティベーションする

In [5]:
import qnexus as qnx
from pytket import Circuit
from datetime import datetime

In [6]:
#qnx.login()
qnx.login_with_credentials()

Already logged in. Tokens are valid.


In [7]:
#projectsを作成/確認
#my_project_ref = qnx.projects.get_or_create("My Project")
all_my_projects = qnx.projects.get_all()
all_my_projects.df()

,name,description,created,modified,contents_modified,archived,id
0,My Project,None,2026-08-24 06:11:30.337257+00:00,2026-08-24 06:11:30.337257+00:00,2026-08-24 07:28:35.659034+00:00,False,e0fbcb19-388b-4225-b938-8ed4f714e6fd


### 2-2. TKET 量子回路をNexus上のQuantinuumのエミュレータで計算

#### Nexusのprojectsを指定/作成し、projectsをアクティベーション

In [8]:
my_project_ref = qnx.projects.get_or_create("My Project")
qnx.context.set_active_project(my_project_ref)

#### 作成した量子回路をprojectsに保存

In [9]:
#Nexusのアクティベーションしたプロジェクトに量子回路を保存
my_circ = qnx.circuits.upload(name="my_circ", circuit=bell)

In [10]:
#保存した量子回路を表示
my_circ.download_circuit()

[H q[0]; CX q[0], q[1]; Measure q[0] --> c[0]; Measure q[1] --> c[1]; ]

In [11]:
from pytket.circuit.display import render_circuit_jupyter
render_circuit_jupyter(my_circ.download_circuit())

#### 作成した量子回路を量子デバイスで実行するために最適化(コンパイル)

In [12]:
#ジョブを区別するための時刻を用意
my_job_name_prefix = datetime.now()

In [13]:
my_comp = qnx.start_compile_job(
    name = f"my_circ compilation {my_job_name_prefix}",
    programs=[my_circ],#複数の量子回路をコンパイルすることが可能 ex. programs=[my_circ1,my_circ2]
    optimisation_level=3,
    backend_config = qnx.QuantinuumConfig(device_name="H2-1LE"),
#    backend_config=qnx.QuantinuumConfig(device_name="H2-Emulator"),
#    project=my_project_ref,
    )

In [14]:
#my_comp:コンパイル前後の量子回路、どのprojectで実行したなどの情報をもったデータフレーム
my_comp.df()

,name,description,created,modified,job_type,last_status,project,backend_config,system,cost,id
0,my_circ compilation 2026-08-24 07:29:10.855445,,2026-08-24 07:29:11.838980+00:00,2026-08-24 07:29:11.838980+00:00,JobType.COMPILE,JobStatusEnum.SUBMITTED,My Project,QuantinuumConfig,Unknown,None,1442d3b6-a9c0-4a40-b5a2-eacb6b8a7187


#### コンパイルジョブの詳細を確認

In [18]:
qnx.jobs.status(my_comp)
#qnx.jobs.wait_for(my_comp)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 8, 24, 7, 29, 32, 234441, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 8, 24, 7, 29, 12, 570246, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 8, 24, 7, 29, 11, 852589, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 8, 24, 7, 29, 28, 554869, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)

In [19]:
#qnx.jobs.resultsを使ってコンパイルの詳細を確認できる
compile_job_result_refs = qnx.jobs.results(my_comp)
compile_job_result_refs.df()

,name,description,created,modified,project,id,job_item_id,job_item_integer_id
0,my_circ-compilation,,2026-08-24 07:29:32.173740+00:00,2026-08-24 07:29:32.205468+00:00,My Project,5c4eb285-89e4-46b9-bd15-6e2fa4ba950f,None,2576705


In [20]:
# コンパイル前の量子回路（オリジナルの量子回路）
circ_input = compile_job_result_refs[0].get_input().download_circuit()
circ_input

[H q[0]; CX q[0], q[1]; Measure q[0] --> c[0]; Measure q[1] --> c[1]; ]

In [21]:
render_circuit_jupyter(circ_input)

In [22]:
# コンパイル後の量子回路（H2-1LEで実行できる量子回路）
circ_output = compile_job_result_refs[0].get_output().download_circuit()
circ_output

[PhasedX(3.5, 0.5) q[0]; PhasedX(2.5, 0.5) q[1]; ZZPhase(0.5) q[0], q[1]; Measure q[0] --> c[0]; PhasedX(0.5, 0) q[1]; Measure q[1] --> c[1]; ]

In [23]:
render_circuit_jupyter(circ_output)

#### エミュレータ（H2-1LE）に量子回路を実行

In [24]:
compiled_circuits = [item.get_output() for item in qnx.jobs.results(my_comp)]
my_exe = qnx.start_execute_job(
    programs=compiled_circuits,
    name=f"my_circ execution {my_job_name_prefix}",
    n_shots=[100] * len(compiled_circuits),
    backend_config=qnx.QuantinuumConfig(device_name="H2-1LE"), #ノイズなしエミュレータ
#    backend_config=qnx.QuantinuumConfig(device_name="H2-Emulator"), #ノイズありエミュレータ
#    project=my_project_ref,
)

#### 実行ジョブの詳細を確認

In [29]:
qnx.jobs.status(my_exe)
#qnx.jobs.wait_for(my_exe)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 8, 24, 7, 30, 6, 43226, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 8, 24, 7, 30, 5, 630440, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 8, 24, 7, 29, 39, 896147, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 8, 24, 7, 30, 5, 919795, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)

In [30]:
execute_job_result_refs = qnx.jobs.results(my_exe)

In [31]:
execute_job_result_refs.df()

,name,description,created,modified,project,id,result_type,cost,job_item_id,job_item_integer_id
0,my_circ execution 2026-08-24 07:29:10.855445,,2026-08-24 07:29:39.884406+00:00,2026-08-24 07:29:39.884406+00:00,My Project,45043969-7acf-46a3-8143-78144fd4a69a,ResultType.PYTKET,None,None,4823630


In [32]:
#実行結果
result = execute_job_result_refs[0].download_result()
result.get_counts()

Counter({(1, 1): 51, (0, 0): 49})

In [33]:
# 実行ジョブにも実行した量子回路のデータはある
circ_exe = execute_job_result_refs[0].get_input().download_circuit()
circ_exe

[PhasedX(3.5, 0.5) q[0]; PhasedX(2.5, 0.5) q[1]; ZZPhase(0.5) q[0], q[1]; Measure q[0] --> c[0]; PhasedX(0.5, 0) q[1]; Measure q[1] --> c[1]; ]

In [34]:
render_circuit_jupyter(circ_exe)

### 2-3. 過去の実行結果を参照

In [35]:
#pandas dataframe
job_refs = qnx.jobs.get_all().df()

In [36]:
job_refs[job_refs['created']>'2026-08-01']

,name,description,created,modified,job_type,last_status,project,backend_config,system,cost,id
0,my_circ compilation 2026-08-24 06:12:01.532120,,2026-08-24 06:12:03.159381+00:00,2026-08-24 06:12:12.915646+00:00,JobType.COMPILE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-1LE,None,7e43776b-9dec-463d-a23a-161ecd2b0e87
1,my_circ compilation 2026-08-24 06:14:15.682307,,2026-08-24 06:14:16.798058+00:00,2026-08-24 06:14:25.467770+00:00,JobType.COMPILE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-1LE,None,919ce9fa-7bbc-4208-9c04-0bc6a94f5626
2,my_circ execution 2026-08-24 06:14:15.682307,,2026-08-24 06:18:09.396845+00:00,2026-08-24 06:18:37.565192+00:00,JobType.EXECUTE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-1LE,None,e5383a5d-e7f1-4e56-bb8a-37e76ad40593
3,my_circ execution 2026-08-24 06:14:15.682307,,2026-08-24 06:18:38.311414+00:00,2026-08-24 06:19:07.425669+00:00,JobType.EXECUTE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-1LE,None,bd20db01-ba3b-4cea-b5d3-b9c4da70003b
4,my_circ execution 2026-08-24 06:14:15.682307,,2026-08-24 06:34:01.270255+00:00,2026-08-24 06:34:07.244216+00:00,JobType.EXECUTE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-1LE,None,2a406a01-af1d-452d-9afb-a3592158ec9a
5,my_circ execution 2026-08-24 06:14:15.682307,,2026-08-24 07:28:35.655703+00:00,2026-08-24 07:29:05.903131+00:00,JobType.EXECUTE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-1LE,None,b347983d-593f-467a-adb5-6aa8c26be337
6,my_circ compilation 2026-08-24 07:29:10.855445,,2026-08-24 07:29:11.838980+00:00,2026-08-24 07:29:32.234441+00:00,JobType.COMPILE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-1LE,None,1442d3b6-a9c0-4a40-b5a2-eacb6b8a7187
7,my_circ execution 2026-08-24 07:29:10.855445,,2026-08-24 07:29:39.884406+00:00,2026-08-24 07:30:06.043226+00:00,JobType.EXECUTE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-1LE,None,a43c114e-3eed-4838-b8d2-dc0020bbf38c


In [37]:
#CompileJobRef
job_comp = qnx.jobs.get(id='7e43776b-9dec-463d-a23a-161ecd2b0e87')
job_comp

CompileJobRef(id=UUID('7e43776b-9dec-463d-a23a-161ecd2b0e87'), annotations=Annotations(name='my_circ compilation 2026-08-24 06:12:01.532120', description='', properties=OrderedDict(), created=datetime.datetime(2026, 8, 24, 6, 12, 3, 159381, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 8, 24, 6, 12, 12, 915646, tzinfo=TzInfo(0))), job_type=<JobType.COMPILE: 'compile'>, last_status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, last_message='The job is completed.', last_status_detail=JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 8, 24, 6, 12, 12, 915646, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 8, 24, 6, 12, 12, 544955, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 8, 24, 6, 12, 3, 171369, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 8, 24, 6, 12, 12, 690993, tzinfo=datetime.timezone.utc), cancelled_time=None, error

In [38]:
compile_job_result_refs1 = qnx.jobs.results(job_comp)
compile_job_result_refs1[0].get_input().download_circuit()

[H q[0]; CX q[0], q[1]; Measure q[0] --> c[0]; Measure q[1] --> c[1]; ]

In [39]:
compile_job_result_refs1[0].get_output().download_circuit()

[PhasedX(3.5, 0.5) q[0]; PhasedX(2.5, 0.5) q[1]; ZZPhase(0.5) q[0], q[1]; Measure q[0] --> c[0]; PhasedX(0.5, 0) q[1]; Measure q[1] --> c[1]; ]

In [40]:
#ExecuteJobRef
job_exe = qnx.jobs.get(id='bd20db01-ba3b-4cea-b5d3-b9c4da70003b')
job_exe

ExecuteJobRef(id=UUID('bd20db01-ba3b-4cea-b5d3-b9c4da70003b'), annotations=Annotations(name='my_circ execution 2026-08-24 06:14:15.682307', description='', properties=OrderedDict(), created=datetime.datetime(2026, 8, 24, 6, 18, 38, 311414, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 8, 24, 6, 19, 7, 425669, tzinfo=TzInfo(0))), job_type=<JobType.EXECUTE: 'execute'>, last_status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, last_message='The job is completed.', last_status_detail=JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 8, 24, 6, 19, 7, 425669, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 8, 24, 6, 19, 7, 179072, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 8, 24, 6, 18, 38, 317857, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 8, 24, 6, 19, 7, 321055, tzinfo=datetime.timezone.utc), cancelled_time=None, error_tim

In [41]:
execute_job_result_refs1 = qnx.jobs.results(job_exe)
execute_job_result_refs1[0].get_input().download_circuit()

[PhasedX(3.5, 0.5) q[0]; PhasedX(2.5, 0.5) q[1]; ZZPhase(0.5) q[0], q[1]; Measure q[0] --> c[0]; PhasedX(0.5, 0) q[1]; Measure q[1] --> c[1]; ]

In [42]:
result = execute_job_result_refs1[0].download_result()
result.get_counts()

Counter({(0, 0): 61, (1, 1): 39})

## 3. 量子回路の解析、ハードウェアで実行な可能な量子回路なのかをシンタックスチェッカーで確認

In [52]:
#以下の`CompileJobRef`に対して解析を行う。
job_ref = qnx.jobs.get(id='7e43776b-9dec-463d-a23a-161ecd2b0e87')  
job_comp = qnx.jobs.results(job_ref)[0].get_output()  
job_comp

CircuitRef(id=UUID('f67341dd-4925-44f5-9fa0-feae4f2567ff'), annotations=Annotations(name='my_circ-QuantinuumBackend-final', description=None, properties=OrderedDict(), created=datetime.datetime(2026, 8, 24, 6, 12, 12, 756847, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 8, 24, 6, 12, 12, 779817, tzinfo=TzInfo(0))), project=ProjectRef(id=UUID('e0fbcb19-388b-4225-b938-8ed4f714e6fd'), annotations=Annotations(name='My Project', description=None, properties=OrderedDict(), created=datetime.datetime(2026, 8, 24, 6, 11, 30, 337257, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 8, 24, 6, 11, 30, 337257, tzinfo=TzInfo(0))), contents_modified=datetime.datetime(2026, 8, 24, 7, 43, 13, 232476, tzinfo=TzInfo(0)), archived=False, type='ProjectRef'), type='CircuitRef')

### 3.1 量子回路の解析


In [53]:
circ = job_comp.download_circuit()
render_circuit_jupyter(circ)

In [65]:
print("# of qubits:" f'{circ.n_qubits}')
print("# of gates:" f'{circ.n_gates}')
print("# of 1qb gates:" f'{circ.n_1qb_gates()}')
print("# of 2qb gates:" f'{circ.n_2qb_gates()}')
print("circuit depth:" f'{circ.depth()}')
print("circuit 2qb gates depth:" f'{circ.depth_2q()}')


# of qubits:2
# of gates:6
# of 1qb gates:3
# of 2qb gates:1
circuit depth:4
circuit 2qb gates depth:1


### 3.2 Syntax checkerの利用 H2利用の前には必ず実行ください。  
#### (Jobが問題なく実行可能か、HQCコストなどの確認)

In [49]:
device_name = "H2-1SC"
config = qnx.QuantinuumConfig(device_name=device_name)
job_name = f"execution-job-qir-{datetime.now()}"
ref_execute_job = qnx.start_execute_job(
    programs=[job_comp],
    n_shots=[1000],
    backend_config=config,
    name=job_name,
)

qnx.jobs.wait_for(ref_execute_job)

JobError: Job errored with detail: Submission error: You do not have access to this machine (code: 14)

In [50]:
qnx.client.circuits.cost(job_comp,n_shots=1000,backend_config=config)

JobError: Job errored with detail: Submission error: You do not have access to this machine (code: 14)

# 弊社Quantinuumのご紹介
- Website（ 英語 ）： https://www.quantinuum.com/
- ウェブサイト（ 日本語 ）： https://quantinuum.co.jp/
- Press Releases（ 英語 ）： https://www.quantinuum.com/news/news#press-release
- ニュース（ 日本語 ）： https://quantinuum.co.jp/news/
- X（ 日本語 ）： https://x.com/quantinuum_jp
- 採用情報（ 英語 ）：https://www.quantinuum.com/careers


## おまけ. 量子回路の変換
pytketでは
- qiskitで記述した量子回路(`qiskit.QuantumCircuit`)からTKETの量子回路のクラスに変換が可能
- TKETで記述した量子回路からqiskitの量子回路(`qiskit.QuantumCircuit`)のクラスに変換が可能

参照：[pytket-qiskit](https://docs.quantinuum.com/tket/extensions/pytket-qiskit/) 